# VITRA Inference-Only Ablation

This notebook runs low-cost component ablations on the EPIC clean test split without retraining. It keeps the same checkpoint and sampler split, then changes only inference inputs such as state and FOV.

## Plan

1. Run smoke ablations on a small clean EPIC subset.
2. Inspect which ablations produce a visible loss gap.
3. Run full EPIC test-split eval only for the useful settings.

The clean split parameters match the previous main eval: `CUTOFF=128000`, `seen_sampler_steps=128000`, and `exclude_seen_ids=True`.

In [ ]:
from pathlib import Path
import json
import os
import shutil
import subprocess
import sys
import textwrap
import pandas as pd

RESOURCE_ROOT = Path("/kaggle/input/notebooks/ldthanh/prepare-vitra-resources")
RESOURCE_REPO = RESOURCE_ROOT / "VITRA"
RESOURCE_WHEELS = RESOURCE_ROOT / "wheels"
WORK_REPO = Path("/kaggle/working/VITRA")

if Path("/kaggle/working").exists():
    if not WORK_REPO.exists():
        assert RESOURCE_REPO.exists(), f"Missing prepared VITRA repo: {RESOURCE_REPO}. Attach prepare-vitra-resources as a Kaggle input."
        shutil.copytree(RESOURCE_REPO, WORK_REPO)
    os.chdir(WORK_REPO)
    repo = WORK_REPO
else:
    repo = Path.cwd()
    assert (repo / "scripts" / "evaluate_pretrained_loss.py").exists(), "Run locally from the repo root."

print("Repo:", repo)
print("Resource root exists:", RESOURCE_ROOT.exists())
print("Offline wheels exists:", RESOURCE_WHEELS.exists())


In [ ]:
%%bash
set -e
if [ -d /kaggle/input/notebooks/ldthanh/prepare-vitra-resources/wheels ]; then
  uv pip install scipy     "torch"     "torchvision"     "torchaudio"     "PyYAML==6.0.2"     "hydra-colorlog>=1.2.0"     "hydra-core>=1.1.1"     "deepspeed==0.16.5"     "tensorboard>=2.13.0"     "tensorboardX>=2.6.2"     "tqdm>=4.65.0"     "transformers==4.47.1"     "diffusers>=0.31.0"     "wandb>=0.19.0"     "numpy<2.0"     "sentence-transformers==2.2.2"     "open_clip_torch==2.20.0"     "datasets==2.12.0"     "draccus==0.8.0"     "einops"     "huggingface_hub"     "json-numpy"     "jsonlines"     "matplotlib"     "peft==0.11.1"     "protobuf"     "rich"     "sentencepiece==0.1.99"     "timm==0.9.10"     "tokenizers>=0.21"     "decord"     "scikit-image"     "brotli"     "imageio-ffmpeg"     "imageio"     "ffmpeg-python"     "opencv-python"     "pympler"     "ultralytics"     "pytorch-lightning"     "yacs"     "utils3d"     --no-index --find-links "/kaggle/input/notebooks/ldthanh/prepare-vitra-resources/wheels"
else
  echo "[INFO] Offline wheels not found; assuming dependencies are already installed."
fi


In [ ]:
from pathlib import Path

EVAL_SCRIPT = '"""\nEvaluate a VITRA weights.pt checkpoint on a deterministic held-out sampler slice.\n\nThis script intentionally mirrors scripts/train.py for model construction and data\nmaterialization, but runs forward-only loss evaluation from a chosen sampler step.\nUse it to evaluate samples that were not consumed during a partial epoch training run.\n"""\n\nimport argparse\nimport copy\nimport json\nimport os\nimport sys\nfrom pathlib import Path\nfrom typing import Any, Dict\n\nimport numpy as np\nimport torch\nimport torch.distributed as dist\nfrom torch.utils.data import DataLoader\nfrom tqdm import tqdm\n\nsys.path.insert(0, os.path.join(os.path.dirname(__file__), ".."))\n\nfrom vitra.datasets.materialize import get_vla_dataset_and_collator\nfrom vitra.datasets.data_mixture import HAND_MIXTURES\nfrom vitra.models.vla_builder import build_vla, load_vla_checkpoint\nfrom vitra.utils import set_global_seed, setup_seed\nfrom vitra.utils.config_utils import load_config\nfrom vitra.utils.overwatch import initialize_overwatch\n\n\nos.environ["TOKENIZERS_PARALLELISM"] = "false"\ntorch.backends.cuda.matmul.allow_tf32 = True\ntorch.backends.cudnn.allow_tf32 = True\n\noverwatch = initialize_overwatch(__name__)\n\n\nclass FixedBatchSampler:\n    def __init__(self, batches):\n        self.batches = batches\n\n    def __iter__(self):\n        yield from self.batches\n\n    def __len__(self):\n        return len(self.batches)\n\n\ndef move_to_device(value: Any, device: torch.device) -> Any:\n    if torch.is_tensor(value):\n        return value.to(device, non_blocking=True)\n    if isinstance(value, dict):\n        return {k: move_to_device(v, device) for k, v in value.items()}\n    if isinstance(value, list):\n        return [move_to_device(v, device) for v in value]\n    if isinstance(value, tuple):\n        return tuple(move_to_device(v, device) for v in value)\n    return value\n\n\ndef tensor_to_float(value: Any) -> float:\n    if torch.is_tensor(value):\n        return float(value.detach().float().mean().cpu().item())\n    return float(value)\n\n\ndef posix_to_str(value: Any) -> Any:\n    if isinstance(value, dict):\n        return {k: posix_to_str(v) for k, v in value.items()}\n    if isinstance(value, list):\n        return [posix_to_str(v) for v in value]\n    if isinstance(value, Path):\n        return str(value)\n    return value\n\n\ndef resolve_weights_path(path: str) -> str:\n    weights_path = Path(path)\n    if weights_path.is_dir():\n        weights_path = weights_path / "weights.pt"\n    if not weights_path.exists():\n        raise FileNotFoundError(f"Checkpoint weights not found: {weights_path}")\n    return str(weights_path)\n\n\ndef cyclic_shift_batch(value: torch.Tensor) -> torch.Tensor:\n    if value.shape[0] <= 1:\n        return value\n    return torch.roll(value, shifts=1, dims=0)\n\n\ndef apply_inference_ablation(batch: Dict[str, Any], args: argparse.Namespace) -> Dict[str, Any]:\n    """Apply input-only ablations for forward-loss evaluation."""\n    state_mode = args.ablate_state\n    if state_mode == "zero_state":\n        batch["current_state"] = torch.zeros_like(batch["current_state"])\n    elif state_mode == "no_state":\n        batch["current_state"] = torch.zeros_like(batch["current_state"])\n        batch["current_state_mask"] = torch.zeros_like(batch["current_state_mask"], dtype=torch.bool)\n    elif state_mode == "shuffle_state":\n        batch["current_state"] = cyclic_shift_batch(batch["current_state"])\n        batch["current_state_mask"] = cyclic_shift_batch(batch["current_state_mask"])\n\n    fov_mode = args.ablate_fov\n    if fov_mode == "zero_fov":\n        batch["fov"] = torch.zeros_like(batch["fov"])\n    elif fov_mode == "shuffle_fov":\n        batch["fov"] = cyclic_shift_batch(batch["fov"])\n\n    return batch\n\n\ndef update_configs(configs: Dict[str, Any], args: argparse.Namespace) -> Dict[str, Any]:\n    configs = copy.deepcopy(configs)\n\n    if args.seed is not None:\n        configs["seed"] = args.seed\n    if args.data_mix is not None:\n        configs["train_dataset"]["data_mix"] = args.data_mix\n    if args.batch_size is not None:\n        configs["batch_size"] = args.batch_size\n    if args.total_batch_size is not None:\n        configs["total_batch_size"] = args.total_batch_size\n    if args.fwd_pred_next_n is not None:\n        configs["fwd_pred_next_n"] = args.fwd_pred_next_n\n    if args.repeated_diffusion_steps is not None:\n        configs["repeated_diffusion_steps"] = args.repeated_diffusion_steps\n    if args.use_bf16:\n        configs["use_bf16"] = True\n\n    configs["output_root"] = Path(configs["output_root"])\n    configs["log_root"] = Path(configs["log_root"])\n    configs["cache_root"] = Path(configs["cache_root"]) / configs["model"]\n    if args.weights != "__dry_run_placeholder__":\n        configs["model_load_path"] = resolve_weights_path(args.weights)\n    else:\n        configs["model_load_path"] = None  # dry_run: no checkpoint needed\n\n    if not args.keep_train_augmentation:\n        train_dataset = configs["train_dataset"]\n        train_dataset["augmentation"] = False\n        train_dataset["state_mask_prob"] = 0.0\n        train_dataset["set_none_ratio"] = 0.0\n\n    return configs\n\n\ndef collect_sampler_ids(batch_sampler, epoch: int, start_step: int, num_steps: int) -> set:\n    batch_sampler.set_epoch(epoch, start_step)\n    ids = set()\n    for step_idx, batch in zip(range(num_steps), batch_sampler):\n        ids.update(tuple(index) for index in batch)\n    return ids\n\n\ndef count_available_samples(\n    batch_sampler,\n    epoch: int,\n    cutoff_step: int,\n    num_datasets: int,\n    seen_ids: set = None,\n    exclude_seen: bool = True,\n) -> Dict[int, int]:\n    """Count unique unseen samples available per dataset from cutoff_step to end-of-epoch.\n\n    Returns a dict mapping dataset_id -> count of qualifying unique samples.\n    This runs without loading the model and is fast enough to sweep across checkpoints.\n    """\n    batch_sampler.set_epoch(epoch, cutoff_step)\n    seen_ids = seen_ids or set()\n    per_dataset_seen: Dict[int, set] = {i: set() for i in range(num_datasets)}\n    counts: Dict[int, int] = {i: 0 for i in range(num_datasets)}\n\n    for mixed_batch in batch_sampler:\n        for index in mixed_batch:\n            index = tuple(index)\n            dataset_id = index[0]\n            if dataset_id not in counts:\n                continue\n            if exclude_seen and index in seen_ids:\n                continue\n            if index in per_dataset_seen[dataset_id]:\n                continue\n            per_dataset_seen[dataset_id].add(index)\n            counts[dataset_id] += 1\n\n    return counts\n\n\ndef get_dataset_names(vla_dataset) -> list:\n    names = []\n    for dataset in vla_dataset.datasets:\n        names.append(getattr(dataset, "dataset_name", f"dataset_{len(names)}"))\n    return names\n\n\ndef get_config_dataset_names(configs: Dict[str, Any]) -> list:\n    data_mix = configs["train_dataset"]["data_mix"]\n    if data_mix in HAND_MIXTURES:\n        return [dataset_name for dataset_name, _ in HAND_MIXTURES[data_mix]]\n    return [data_mix]\n\n\ndef collect_dataset_batches(\n    batch_sampler,\n    epoch: int,\n    start_step: int,\n    dataset_id: int,\n    num_batches: int,\n    batch_size: int,\n    exclude_ids: set = None,\n    unique: bool = True,\n) -> list:\n    """Filter the mixed sampler stream to fixed-size batches from one dataset id."""\n    batch_sampler.set_epoch(epoch, start_step)\n    batches = []\n    current_batch = []\n    exclude_ids = exclude_ids or set()\n    used_ids = set()\n\n    for mixed_batch in batch_sampler:\n        for index in mixed_batch:\n            index = tuple(index)\n            if index[0] != dataset_id:\n                continue\n            if index in exclude_ids:\n                continue\n            if unique and index in used_ids:\n                continue\n\n            current_batch.append(index)\n            used_ids.add(index)\n            if len(current_batch) == batch_size:\n                batches.append(current_batch)\n                current_batch = []\n                if len(batches) == num_batches:\n                    return batches\n\n    raise RuntimeError(\n        f"Could only collect {len(batches)} batches for dataset_id={dataset_id}; "\n        f"requested {num_batches}. Try a smaller --eval_batches, earlier --eval_sampler_step, "\n        f"or disable --unique_per_dataset_eval."\n    )\n\n\ndef make_dataset_and_sampler(configs: Dict[str, Any], processor, args: argparse.Namespace):\n    batch_size = configs["batch_size"]\n    train_dataset = configs["train_dataset"]\n\n    vla_dataset, collator, batch_sampler = get_vla_dataset_and_collator(\n        train_dataset["data_root_dir"],\n        train_dataset["data_mix"],\n        augmentation=train_dataset["augmentation"],\n        shard_num=overwatch.world_size(),\n        shard_index=overwatch.rank(),\n        seed=configs["seed"],\n        future_action_window_size=configs["fwd_pred_next_n"] - 1,\n        processor=processor,\n        batch_size=batch_size,\n        normalization=train_dataset.get("normalization", True),\n        flip_augmentation=train_dataset.get("flip_augmentation", 1.0),\n        set_none_ratio=train_dataset.get("set_none_ratio", 0.0),\n        action_type=train_dataset.get("action_type", "angle"),\n        use_rel=train_dataset.get("use_rel", False),\n        rel_mode=train_dataset.get("rel_mode", "step"),\n        clip_len=train_dataset.get("clip_len", None),\n        state_mask_prob=train_dataset.get("state_mask_prob", 0.0),\n        target_image_height=train_dataset.get("target_image_height", 224),\n    )\n\n    setup_seed(configs["seed"], rank=overwatch.rank())\n    return vla_dataset, collator, batch_sampler\n\n\ndef make_dataloader(vla_dataset, collator, batch_sampler, args: argparse.Namespace, configs: Dict[str, Any]):\n    train_dataset = configs["train_dataset"]\n    num_workers = args.num_workers\n    if num_workers is None:\n        num_workers = train_dataset["num_workers"]\n    prefetch_factor = args.prefetch_factor\n    if prefetch_factor is None:\n        prefetch_factor = train_dataset["prefetch_factor"]\n    if num_workers == 0 or prefetch_factor == 0:\n        prefetch_factor = None\n\n    dataloader = DataLoader(\n        vla_dataset,\n        batch_sampler=batch_sampler,\n        collate_fn=collator,\n        num_workers=num_workers,\n        prefetch_factor=prefetch_factor,\n        worker_init_fn=set_global_seed(configs["seed"], get_worker_init_fn=True),\n        persistent_workers=num_workers > 0,\n        pin_memory=num_workers > 0,\n    )\n    return dataloader, batch_sampler\n\n\n@torch.no_grad()\ndef evaluate(configs: Dict[str, Any], args: argparse.Namespace, dataset_name: str = None) -> Dict[str, Any]:\n    device = torch.device(f"cuda:{overwatch.local_rank()}" if torch.cuda.is_available() else "cpu")\n    if torch.cuda.is_available():\n        torch.cuda.set_device(device)\n        torch.cuda.empty_cache()\n\n    model = build_vla(configs=configs)\n    model = load_vla_checkpoint(model, configs["model_load_path"])\n    model.model.use_bf16 = configs["use_bf16"]\n    model.use_bf16 = configs["use_bf16"]\n    model = model.to(device).eval()\n\n    vla_dataset, collator, batch_sampler = make_dataset_and_sampler(configs, model.processor, args)\n    dataset_names = get_dataset_names(vla_dataset)\n\n    overlap_count = None\n    seen_ids = None\n    if args.seen_sampler_steps is not None:\n        seen_ids = collect_sampler_ids(batch_sampler, args.seen_epoch, 0, args.seen_sampler_steps)\n\n    if dataset_name is None:\n        if seen_ids is not None:\n            eval_ids = collect_sampler_ids(batch_sampler, args.eval_epoch, args.eval_sampler_step, args.eval_batches)\n            overlap = seen_ids & eval_ids\n            overlap_count = len(overlap)\n            if overlap_count > 0 and not args.allow_seen_overlap:\n                raise RuntimeError(\n                    f"Evaluation sampler slice overlaps the seen sampler slice by {overlap_count} sample ids. "\n                    f"This can happen with weighted oversampling. Increase --eval_sampler_step or pass "\n                    f"--allow_seen_overlap for a weighted reference-set comparison."\n                )\n        batch_sampler.set_epoch(args.eval_epoch, args.eval_sampler_step)\n        eval_batch_sampler = batch_sampler\n        eval_dataset_id = None\n    else:\n        if dataset_name not in dataset_names:\n            raise ValueError(f"Unknown dataset {dataset_name!r}. Available datasets: {dataset_names}")\n        eval_dataset_id = dataset_names.index(dataset_name)\n        fixed_batches = collect_dataset_batches(\n            batch_sampler,\n            args.eval_epoch,\n            args.eval_sampler_step,\n            eval_dataset_id,\n            args.eval_batches,\n            configs["batch_size"],\n            exclude_ids=seen_ids if args.exclude_seen_ids else None,\n            unique=args.unique_per_dataset_eval,\n        )\n        eval_batch_sampler = FixedBatchSampler(fixed_batches)\n\n        if args.seen_sampler_steps is not None:\n            eval_ids = {index for batch in fixed_batches for index in batch}\n            overlap_count = len(seen_ids & eval_ids)\n            if overlap_count > 0:\n                raise RuntimeError(\n                    f"Per-dataset evaluation for {dataset_name} overlaps the seen sampler slice by "\n                    f"{overlap_count} sample ids. Increase --eval_sampler_step."\n                )\n\n    dataloader, _ = make_dataloader(vla_dataset, collator, eval_batch_sampler, args, configs)\n\n    output_jsonl = Path(args.output_jsonl)\n    if dataset_name is not None:\n        output_jsonl = output_jsonl.with_name(f"{output_jsonl.stem}.{dataset_name}{output_jsonl.suffix}")\n    output_jsonl.parent.mkdir(parents=True, exist_ok=True)\n\n    component_sums: Dict[str, float] = {}\n    component_values: Dict[str, list] = {}\n    num_batches = 0\n    num_examples = 0\n\n    progress = tqdm(\n        zip(range(args.eval_batches), dataloader),\n        total=args.eval_batches,\n        disable=not overwatch.is_rank_zero(),\n        desc="evaluating",\n    )\n    with open(output_jsonl, "w") as f:\n        for local_step, batch in progress:\n            batch = move_to_device(batch, device)\n            batch = apply_inference_ablation(batch, args)\n            prediction = model.forward(\n                batch["pixel_values"],\n                batch["input_ids"],\n                attention_mask=batch["attention_mask"],\n                action_labels=batch["actions"],\n                action_masks=batch["action_masks"],\n                current_state_mask=batch["current_state_mask"],\n                current_state=batch["current_state"],\n                data_source=["action"],\n                fov=batch["fov"],\n            )\n\n            record = {\n                "eval_batch": local_step,\n                "sampler_epoch": args.eval_epoch,\n                "sampler_step": args.eval_sampler_step + local_step if dataset_name is None else None,\n                "source_start_sampler_step": args.eval_sampler_step,\n                "batch_size": int(batch["input_ids"].shape[0]),\n                "dataset": dataset_name or "mixed",\n                "ablate_state": args.ablate_state,\n                "ablate_fov": args.ablate_fov,\n                "repeated_diffusion_steps": configs.get("repeated_diffusion_steps"),\n            }\n            for key, value in prediction.items():\n                record[key] = tensor_to_float(value)\n                component_sums[key] = component_sums.get(key, 0.0) + record[key]\n                component_values.setdefault(key, []).append(record[key])\n\n            num_batches += 1\n            num_examples += record["batch_size"]\n            progress.set_postfix(loss=record.get("loss"))\n            f.write(json.dumps(record) + "\\n")\n\n    summary = {\n        "checkpoint": configs["model_load_path"],\n        "dataset": dataset_name or "mixed",\n        "dataset_id": eval_dataset_id,\n        "dataset_names": dataset_names,\n        "config": posix_to_str(configs),\n        "eval_epoch": args.eval_epoch,\n        "eval_sampler_step": args.eval_sampler_step,\n        "eval_batches": num_batches,\n        "eval_examples": num_examples,\n        "seen_epoch": args.seen_epoch,\n        "seen_sampler_steps": args.seen_sampler_steps,\n        "seen_eval_overlap": overlap_count,\n        "exclude_seen_ids": args.exclude_seen_ids if dataset_name is not None else False,\n        "unique_per_dataset_eval": args.unique_per_dataset_eval if dataset_name is not None else False,\n        "sampler_num_iters": batch_sampler.num_iters,\n        "sampler_dataset_lengths": batch_sampler._dataset_lengths,\n        "sampler_weights": batch_sampler.weights,\n        "ablation": {\n            "ablate_state": args.ablate_state,\n            "ablate_fov": args.ablate_fov,\n            "repeated_diffusion_steps": configs.get("repeated_diffusion_steps"),\n        },\n        "metrics": {},\n    }\n    for key, values in component_values.items():\n        array = np.asarray(values, dtype=np.float64)\n        summary["metrics"][key] = {\n            "mean": float(array.mean()),\n            "std": float(array.std()),\n            "min": float(array.min()),\n            "max": float(array.max()),\n            "p10": float(np.quantile(array, 0.10)),\n            "p50": float(np.quantile(array, 0.50)),\n            "p90": float(np.quantile(array, 0.90)),\n        }\n\n    summary_path = output_jsonl.with_suffix(".summary.json")\n    with open(summary_path, "w") as f:\n        json.dump(summary, f, indent=2)\n\n    del model\n    if torch.cuda.is_available():\n        torch.cuda.empty_cache()\n\n    return summary\n\n\ndef parse_args() -> argparse.Namespace:\n    parser = argparse.ArgumentParser(description="Evaluate VITRA weights.pt loss on a deterministic sampler slice.")\n    parser.add_argument("--config", required=True, type=str, help="Training config JSON/YAML used for the run.")\n    parser.add_argument(\n        "--weights",\n        default=None,\n        type=str,\n        help="Path to weights.pt or a checkpoint directory containing weights.pt. Not required for --dry_run.",\n    )\n    parser.add_argument("--output_jsonl", default=".tmp/eval_loss.jsonl", type=str, help="Path for per-batch JSONL metrics.")\n    parser.add_argument("--eval_dataset", default=None, type=str, help="Evaluate one dataset from the configured mixture, e.g. epic or ssv2.")\n    parser.add_argument("--eval_each_dataset", action="store_true", help="Evaluate each dataset in the configured mixture separately.")\n    parser.add_argument("--eval_epoch", default=0, type=int, help="Sampler epoch to evaluate.")\n    parser.add_argument("--eval_sampler_step", default=20000, type=int, help="Sampler micro-batch step to start evaluation from.")\n    parser.add_argument("--eval_batches", default=200, type=int, help="Number of sampler batches to evaluate.")\n    parser.add_argument("--seen_epoch", default=0, type=int, help="Sampler epoch for the seen slice overlap check.")\n    parser.add_argument("--seen_sampler_steps", default=None, type=int, help="If set, assert eval ids do not overlap sampler steps [0, N).")\n    parser.add_argument("--seen_optimizer_steps", default=None, type=int, help="Convert optimizer steps to seen sampler steps using gradient accumulation.")\n    parser.add_argument("--grad_accumulation_steps", default=None, type=int, help="Override gradient accumulation when using --seen_optimizer_steps.")\n    parser.add_argument(\n        "--allow_seen_overlap",\n        action="store_true",\n        help="Allow raw sample-id overlap in mixed weighted evaluation. Useful because weighted oversampling can repeat ids.",\n    )\n    parser.add_argument(\n        "--exclude_seen_ids",\n        action=argparse.BooleanOptionalAction,\n        default=True,\n        help="For per-dataset evaluation, skip raw sample ids consumed in the seen sampler slice.",\n    )\n    parser.add_argument(\n        "--unique_per_dataset_eval",\n        action=argparse.BooleanOptionalAction,\n        default=True,\n        help="For per-dataset evaluation, avoid duplicate raw sample ids in the eval batches.",\n    )\n    parser.add_argument("--data_mix", default=None, type=str, help="Override train_dataset.data_mix.")\n    parser.add_argument("--seed", default=None, type=int, help="Override config seed.")\n    parser.add_argument("--batch_size", default=None, type=int, help="Override per-device batch size.")\n    parser.add_argument("--total_batch_size", default=None, type=int, help="Override global batch size for metadata consistency.")\n    parser.add_argument("--fwd_pred_next_n", default=None, type=int, help="Override future action horizon.")\n    parser.add_argument("--repeated_diffusion_steps", default=None, type=int, help="Override repeated diffusion steps for forward-loss evaluation.")\n    parser.add_argument("--num_workers", default=None, type=int, help="Override DataLoader workers.")\n    parser.add_argument("--prefetch_factor", default=None, type=int, help="Override DataLoader prefetch factor.")\n    parser.add_argument("--use_bf16", action="store_true", help="Force model use_bf16=True.")\n    parser.add_argument(\n        "--ablate_state",\n        default="none",\n        choices=["none", "zero_state", "no_state", "shuffle_state"],\n        help=(\n            "Inference-only state ablation. zero_state zeros state values but keeps masks; "\n            "no_state zeros values and masks; shuffle_state cyclically shifts states within each batch."\n        ),\n    )\n    parser.add_argument(\n        "--ablate_fov",\n        default="none",\n        choices=["none", "zero_fov", "shuffle_fov"],\n        help="Inference-only FOV ablation. shuffle_fov cyclically shifts FOV values within each batch.",\n    )\n    parser.add_argument(\n        "--keep_train_augmentation",\n        action="store_true",\n        help="Keep train_dataset augmentation/state masking settings. Default disables stochastic eval transforms.",\n    )\n    # ── Dry-run / counting mode ────────────────────────────────────────────────\n    parser.add_argument(\n        "--dry_run",\n        action="store_true",\n        help=(\n            "Count unique unseen samples per dataset for each cutoff in --sweep_cutoff_steps "\n            "without loading the model. For each cutoff C, samples from [0, C) are treated as "\n            "seen (inferred automatically); samples from [C, epoch-end) that are unique and "\n            "not in the seen set are counted. Total corpus size is derived from the dataset "\n            "lengths. --seen_sampler_steps is ignored in this mode."\n        ),\n    )\n    parser.add_argument(\n        "--sweep_cutoff_steps",\n        default=None,\n        type=str,\n        help=(\n            "Comma-separated list of sampler cutoff steps to sweep in --dry_run mode "\n            "(overrides --eval_sampler_step for each entry). "\n            "Example: --sweep_cutoff_steps 64000,80000,96000,112000"\n        ),\n    )\n    return parser.parse_args()\n\n\ndef run_dry_run(configs: Dict[str, Any], args: argparse.Namespace) -> None:\n    """Count unique unseen samples available per dataset from one or more cutoff steps.\n\n    Builds the sampler once (no model load). For each cutoff step C:\n      - seen_ids are collected from sampler steps [0, C) automatically.\n      - count_available_samples walks [C, epoch-end) and counts unique unseen samples.\n      - total_corpus_size is derived from sum(batch_sampler._dataset_lengths).\n\n    Prints a table of (cutoff, total_available, fraction_of_corpus, per_dataset)\n    so you can identify which 16k-step checkpoint boundary leaves ~20% for eval.\n    """\n    import time\n\n    overwatch.info("[dry_run] Building dataset and sampler (no model load) …")\n    t0 = time.monotonic()\n\n    # Build only the sampler; we need a processor to call get_vla_dataset_and_collator.\n    # Construct the model CPU-only just to extract the processor, then discard it.\n    from vitra.models.vla_builder import build_vla  # noqa: F811\n\n    tmp_model = build_vla(configs=copy.deepcopy(configs))\n    processor = tmp_model.processor\n    del tmp_model\n\n    vla_dataset, _collator, batch_sampler = make_dataset_and_sampler(configs, processor, args)\n    dataset_names = get_dataset_names(vla_dataset)\n    num_datasets = len(dataset_names)\n\n    # Total unique corpus size is the sum of per-dataset lengths from the sampler.\n    total_corpus = sum(batch_sampler._dataset_lengths)\n    overwatch.info(\n        f"[dry_run] Sampler ready in {time.monotonic() - t0:.1f}s — "\n        f"{num_datasets} datasets: {dataset_names}, "\n        f"total_corpus={total_corpus:,}"\n    )\n\n    # Determine which cutoff steps to sweep.\n    if args.sweep_cutoff_steps is not None:\n        cutoff_steps = [int(s.strip()) for s in args.sweep_cutoff_steps.split(",")]\n    else:\n        cutoff_steps = [args.eval_sampler_step]\n\n    results = []\n    for cutoff in cutoff_steps:\n        # Seen samples = everything the model was trained on up to this checkpoint.\n        # Inferred directly from the cutoff step; --seen_sampler_steps is ignored here.\n        t1 = time.monotonic()\n        overwatch.info(f"[dry_run] cutoff={cutoff}: collecting seen_ids from [0, {cutoff}) …")\n        seen_ids = collect_sampler_ids(batch_sampler, args.seen_epoch, 0, cutoff)\n        overwatch.info(\n            f"[dry_run] cutoff={cutoff}: {len(seen_ids):,} seen sample-ids "\n            f"in {time.monotonic() - t1:.1f}s"\n        )\n\n        t2 = time.monotonic()\n        counts = count_available_samples(\n            batch_sampler,\n            epoch=args.eval_epoch,\n            cutoff_step=cutoff,\n            num_datasets=num_datasets,\n            seen_ids=seen_ids if args.exclude_seen_ids else set(),\n            exclude_seen=args.exclude_seen_ids,\n        )\n        elapsed = time.monotonic() - t2\n        total_available = sum(counts.values())\n        fraction = round(total_available / total_corpus, 4) if total_corpus > 0 else None\n        entry = {\n            "cutoff_step": cutoff,\n            "seen_ids_count": len(seen_ids),\n            "total_available": total_available,\n            "fraction_of_corpus": fraction,\n            "elapsed_s": round(elapsed, 2),\n            "per_dataset": {dataset_names[i]: counts[i] for i in range(num_datasets)},\n        }\n        results.append(entry)\n        overwatch.info(\n            f"[dry_run] cutoff={cutoff}: available={total_available:,} "\n            f"({fraction:.1%} of corpus) in {elapsed:.1f}s"\n        )\n        for name, cnt in entry["per_dataset"].items():\n            overwatch.info(f"  {name}: {cnt:,}")\n\n    # Write JSON summary.\n    output_path = Path(args.output_jsonl).with_suffix(".dry_run.json")\n    output_path.parent.mkdir(parents=True, exist_ok=True)\n    dry_run_summary = {\n        "mode": "dry_run",\n        "exclude_seen_ids": args.exclude_seen_ids,\n        "dataset_names": dataset_names,\n        "total_corpus": total_corpus,\n        "sampler_num_iters": batch_sampler.num_iters,\n        "sampler_dataset_lengths": batch_sampler._dataset_lengths,\n        "sampler_weights": batch_sampler.weights,\n        "cutoff_sweep": results,\n    }\n    with open(output_path, "w") as f:\n        json.dump(dry_run_summary, f, indent=2)\n    overwatch.info(f"[dry_run] Summary written to {output_path}")\n    if overwatch.is_rank_zero():\n        print(json.dumps(dry_run_summary, indent=2))\n\n\nif __name__ == "__main__":\n    if not dist.is_initialized():\n        os.environ.setdefault("RANK", "0")\n        os.environ.setdefault("WORLD_SIZE", "1")\n        os.environ.setdefault("LOCAL_RANK", "0")\n        os.environ.setdefault("MASTER_ADDR", "localhost")\n        os.environ.setdefault("MASTER_PORT", "29501")\n        backend = "nccl" if torch.cuda.is_available() else "gloo"\n        dist.init_process_group(backend=backend)\n\n    args = parse_args()\n\n    # In dry_run mode --weights is not required.\n    if not args.dry_run and args.weights is None:\n        raise ValueError("--weights is required unless --dry_run is set.")\n\n    # Patch resolve_weights_path to be a no-op when weights not provided.\n    if args.weights is None:\n        args.weights = "__dry_run_placeholder__"\n\n    configs = update_configs(load_config(args.config), args)\n\n    if args.seen_optimizer_steps is not None and args.seen_sampler_steps is None:\n        grad_accumulation_steps = args.grad_accumulation_steps\n        if grad_accumulation_steps is None:\n            grad_accumulation_steps = configs["total_batch_size"] // configs["batch_size"] // overwatch.world_size()\n        args.seen_sampler_steps = args.seen_optimizer_steps * grad_accumulation_steps\n\n    if args.eval_each_dataset and args.eval_dataset is not None:\n        raise ValueError("Use either --eval_each_dataset or --eval_dataset, not both.")\n\n    if args.dry_run:\n        run_dry_run(configs, args)\n    elif args.eval_each_dataset:\n        dataset_names = get_config_dataset_names(configs)\n        summaries = [evaluate(configs, args, dataset_name=name) for name in dataset_names]\n        summary = {\n            "datasets": [item["dataset"] for item in summaries],\n            "summary_paths": [\n                str(Path(args.output_jsonl).with_name(f"{Path(args.output_jsonl).stem}.{item[\'dataset\']}{Path(args.output_jsonl).suffix}").with_suffix(".summary.json"))\n                for item in summaries\n            ],\n            "metrics": {item["dataset"]: item["metrics"] for item in summaries},\n        }\n        if overwatch.is_rank_zero():\n            print(json.dumps({"summary_paths": summary["summary_paths"], "metrics": summary["metrics"]}, indent=2))\n    else:\n        summary = evaluate(configs, args, dataset_name=args.eval_dataset)\n        if overwatch.is_rank_zero():\n            if args.eval_dataset is not None:\n                output_jsonl = Path(args.output_jsonl)\n                summary_path = str(output_jsonl.with_name(f"{output_jsonl.stem}.{args.eval_dataset}{output_jsonl.suffix}").with_suffix(".summary.json"))\n            else:\n                summary_path = str(Path(args.output_jsonl).with_suffix(".summary.json"))\n            print(json.dumps({"summary_path": summary_path, "metrics": summary["metrics"]}, indent=2))\n\n    if dist.is_initialized():\n        dist.barrier()\n        dist.destroy_process_group()\n'

Path("scripts/evaluate_pretrained_loss.py").write_text(EVAL_SCRIPT, encoding="utf-8")
print("Patched scripts/evaluate_pretrained_loss.py with inference-only ablation flags")
print("Contains --ablate_state:", "--ablate_state" in EVAL_SCRIPT)


## Dataset Links

This mirrors the old pretraining notebook: create the expected `data/VITRA_1M` symlinks after copying `VITRA` from `prepare-vitra-resources`.


In [ ]:
%%bash
set -e
mkdir -p data/VITRA_1M/Video data/VITRA_1M/Annotation

# Link original EPIC videos. This mirrors SKIP_PREPROCESSING=True in the old notebook.
mkdir -p data/VITRA_1M/Video/Epic-Kitchen_root
if compgen -G "/kaggle/input/datasets/ldthanh/epic-kitchens-100-p*/*/*/*.MP4" > /dev/null; then
  for vid in /kaggle/input/datasets/ldthanh/epic-kitchens-100-p*/*/*/*.MP4; do
    ln -sf "$vid" "data/VITRA_1M/Video/Epic-Kitchen_root/$(basename "$vid")"
  done
else
  echo "[WARN] EPIC video input pattern not found. Check attached Kaggle datasets."
fi

if [ -d /kaggle/input/datasets/nahidsiddique/something-something-v2/20bn-something-something-v2/20bn-something-something-v2 ]; then
  ln -sfn /kaggle/input/datasets/nahidsiddique/something-something-v2/20bn-something-something-v2/20bn-something-something-v2 data/VITRA_1M/Video/Somethingsomething-v2_root
else
  echo "[WARN] SSV2 video root not found. EPIC-only eval can still run."
fi
if [ -d /kaggle/input/datasets/ldthanh/vitra-1m/epic/epic ]; then
  ln -sfn /kaggle/input/datasets/ldthanh/vitra-1m/epic/epic data/VITRA_1M/Annotation/epic
fi
if [ -d /kaggle/input/datasets/ldthanh/vitra-1m/ssv2/ssv2 ]; then
  ln -sfn /kaggle/input/datasets/ldthanh/vitra-1m/ssv2/ssv2 data/VITRA_1M/Annotation/ssv2
fi
if [ -d /kaggle/input/datasets/ldthanh/vitra-1m/statistics ]; then
  ln -sfn /kaggle/input/datasets/ldthanh/vitra-1m/statistics data/VITRA_1M/Annotation/statistics
fi

find data/VITRA_1M -maxdepth 3 -type l -print | head -100


In [ ]:
CONFIG = "vitra/configs/human_pretrain.json"
WEIGHTS_16K = "/kaggle/input/models/ldthanh/vitra-vla-3b/transformers/magic_mix_epic_ssv2/8/final-epoch=0-step=16000.ckpt/weights.pt"

CUTOFF = 128000
SMOKE_BATCHES = 200      # 12,800 examples with batch_size=64
FULL_BATCHES = 16400     # 1,049,600 examples, about 3h+ per setting in prior runs
NUM_WORKERS = 32

OUT_ROOT = Path("ablation_results")
OUT_ROOT.mkdir(exist_ok=True)

assert Path(CONFIG).exists(), CONFIG
assert Path(WEIGHTS_16K).exists(), WEIGHTS_16K

In [ ]:
def run_eval(name, batches, ablate_state="none", ablate_fov="none", repeated_diffusion_steps=None, phase="smoke"):
    out_dir = OUT_ROOT / phase
    out_dir.mkdir(parents=True, exist_ok=True)
    output_jsonl = out_dir / f"{name}.jsonl"
    cmd = [
        "python", "scripts/evaluate_pretrained_loss.py",
        "--config", CONFIG,
        "--weights", WEIGHTS_16K,
        "--eval_dataset", "epic",
        "--eval_sampler_step", str(CUTOFF),
        "--eval_batches", str(batches),
        "--seen_sampler_steps", str(CUTOFF),
        "--output_jsonl", str(output_jsonl),
        "--num_workers", str(NUM_WORKERS),
        "--ablate_state", ablate_state,
        "--ablate_fov", ablate_fov,
    ]
    if repeated_diffusion_steps is not None:
        cmd += ["--repeated_diffusion_steps", str(repeated_diffusion_steps)]
    print(" ".join(cmd))
    subprocess.run(cmd, check=True)
    return output_jsonl.with_name(f"{output_jsonl.stem}.epic.summary.json")

def load_summaries(phase):
    rows = []
    for path in sorted((OUT_ROOT / phase).glob("*.summary.json")):
        data = json.loads(path.read_text())
        metrics = data["metrics"]
        rows.append({
            "file": path.name,
            "checkpoint": "16k",
            "dataset": data["dataset"],
            "examples": data["eval_examples"],
            "ablate_state": data.get("ablation", {}).get("ablate_state", "none"),
            "ablate_fov": data.get("ablation", {}).get("ablate_fov", "none"),
            "repeated_diffusion_steps": data.get("ablation", {}).get("repeated_diffusion_steps"),
            "loss": metrics["loss"]["mean"],
            "left_hand_6d": metrics["left_hand_6d"]["mean"],
            "right_hand_6d": metrics["right_hand_6d"]["mean"],
            "left_hand_joints": metrics["left_hand_joints"]["mean"],
            "right_hand_joints": metrics["right_hand_joints"]["mean"],
        })
    df = pd.DataFrame(rows)
    if not df.empty:
        df = df.sort_values(["examples", "ablate_state", "ablate_fov", "repeated_diffusion_steps"])
        baseline = df[(df["ablate_state"] == "none") & (df["ablate_fov"] == "none")]
        if not baseline.empty:
            base_loss = baseline.iloc[0]["loss"]
            df["delta_vs_baseline"] = df["loss"] - base_loss
            df["relative_vs_baseline_pct"] = 100 * df["delta_vs_baseline"] / base_loss
    return df


## Smoke Ablation

Run all cheap settings on the same clean EPIC subset. These are the settings to include in the report if full eval time is limited.

In [ ]:
SMOKE_SETTINGS = [
    dict(name="baseline", ablate_state="none", ablate_fov="none"),
    dict(name="zero_state", ablate_state="zero_state", ablate_fov="none"),
    dict(name="no_state", ablate_state="no_state", ablate_fov="none"),
    dict(name="shuffle_state", ablate_state="shuffle_state", ablate_fov="none"),
    dict(name="zero_fov", ablate_state="none", ablate_fov="zero_fov"),
    dict(name="shuffle_fov", ablate_state="none", ablate_fov="shuffle_fov"),
    dict(name="rds1", ablate_state="none", ablate_fov="none", repeated_diffusion_steps=1),
    dict(name="rds4", ablate_state="none", ablate_fov="none", repeated_diffusion_steps=4),
]

for setting in SMOKE_SETTINGS:
    run_eval(batches=SMOKE_BATCHES, phase="smoke", **setting)

smoke_df = load_summaries("smoke")
smoke_df.to_csv(OUT_ROOT / "smoke_summary.csv", index=False)
smoke_df

## Full Test-Split Ablation

Full eval is expensive. Turn `RUN_FULL` on only after smoke results look useful. Suggested minimum: `no_state`; suggested second setting: `shuffle_state`.

In [ ]:
RUN_FULL = False

FULL_SETTINGS = [
    dict(name="no_state_full", ablate_state="no_state", ablate_fov="none"),
    dict(name="shuffle_state_full", ablate_state="shuffle_state", ablate_fov="none"),
    # Uncomment if smoke shows a clear FOV effect and you still have GPU time.
    # dict(name="zero_fov_full", ablate_state="none", ablate_fov="zero_fov"),
]

if RUN_FULL:
    for setting in FULL_SETTINGS:
        run_eval(batches=FULL_BATCHES, phase="full", **setting)

full_df = load_summaries("full")
if not full_df.empty:
    full_df.to_csv(OUT_ROOT / "full_summary.csv", index=False)
full_df

In [ ]:
combined = pd.concat([load_summaries("smoke"), load_summaries("full")], ignore_index=True)
if not combined.empty:
    combined.to_csv(OUT_ROOT / "ablation_summary.csv", index=False)
    print(combined.to_string(index=False))
combined

In [ ]:
%%bash
set -e
zip -qr ../vitra_ablation_results.zip ablation_results
ls -lh ../vitra_ablation_results.zip